# 🐍 Clase 14 · Avellaneda-Stoikov — modelo y simulación

> Sustituir el skew heurístico por el resultado del modelo de Avellaneda-Stoikov (reservation price + optimal spread), y ponerlo a correr: simular, ver cómo controla el inventario y barrer gamma.

**Hoy construyes:** AvellanedaStoikov: reservation price, optimal spread y barridos de gamma.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Abre **💡 Ver solución**.

**Núcleo:** los primeros (en clase) · **Si vamos bien:** el resto · **Más:** el cuaderno de auxiliares.

### 1. Reservation price con inventario

Crea `AvellanedaStoikov('BTC', horizon=500)`, fija `_inventory=3`, `_t=0`. Guarda `r = reservation_price(100)`. Debe ser < 100.

<sub>practicas: fórmula r</sub>

In [ ]:
from exchange.strategies import AvellanedaStoikov
strat = AvellanedaStoikov('BTC', gamma=0.1, sigma=10, kappa=1.5, horizon=500)
strat._inventory = 3
strat._t = 0
r = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert r < 100, 'inventario largo -> r por debajo del mid'
print('ok  r=%.2f' % r)

<details>
<summary>💡 Ver solución</summary>

```python
r = strat.reservation_price(100)
```

</details>

### 2. Optimal spread positivo

Guarda `d = strat.optimal_spread()`. Debe ser positivo.

<sub>practicas: fórmula d</sub>

In [ ]:
from exchange.strategies import AvellanedaStoikov
strat = AvellanedaStoikov('BTC', gamma=0.1, sigma=10, kappa=1.5, horizon=500)
strat._t = 0
d = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert d > 0, 'el spread óptimo es positivo'
print('ok  d=%.4f' % d)

<details>
<summary>💡 Ver solución</summary>

```python
d = strat.optimal_spread()
```

</details>

### 3. Cotizaciones A-S

Con inventario 0, las cotizaciones deben estar centradas en el mid. Guarda `bid`, `ask` con un libro de mid 100.

<sub>practicas: quotes simétricas en torno a r</sub>

In [ ]:
from exchange import OrderBook, Level
from exchange.strategies import AvellanedaStoikov
strat = AvellanedaStoikov('BTC', gamma=0.1, sigma=10, kappa=1.5, horizon=500)
strat._t = 0
book = OrderBook('BTC', [Level(99.5,1)], [Level(100.5,1)])
bid = None
ask = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert bid < 100 < ask
assert abs((bid + ask)/2 - 100) < 1e-6, 'inventario 0 -> centradas en el mid'
print('ok  bid=%.3f ask=%.3f' % (bid, ask))

<details>
<summary>💡 Ver solución</summary>

```python
bid, ask = strat.quotes(book)
```

</details>

### 4. Inventario inclina el centro

Con `_inventory=5`, el centro `(bid+ask)/2` debe quedar por debajo del mid (inclina para soltar). Guarda `center`.

<sub>practicas: A-S vs naive</sub>

In [ ]:
from exchange import OrderBook, Level
from exchange.strategies import AvellanedaStoikov
strat = AvellanedaStoikov('BTC', gamma=0.1, sigma=10, kappa=1.5, horizon=500)
strat._t = 0
strat._inventory = 5
book = OrderBook('BTC', [Level(99.5,1)], [Level(100.5,1)])
center = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert center < 100, 'largo -> centro por debajo del mid'
print('ok  center=%.2f' % center)

<details>
<summary>💡 Ver solución</summary>

```python
bid, ask = strat.quotes(book)
center = (bid + ask) / 2
```

</details>

### 5. Simula el A-S

Simula `AvellanedaStoikov('BTC', gamma=0.1, sigma=0.5, horizon=400)` con `steps=400`. Guarda `max_inv` y `pnl`.

<sub>practicas: MMSimulation con A-S</sub>

In [ ]:
from exchange.strategies import AvellanedaStoikov
from exchange.simulation import MMSimulation
max_inv = None
pnl = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert max_inv >= 0 and isinstance(pnl, float)
print('ok  max|inv|=%.2f pnl=%.2f' % (max_inv, pnl))

<details>
<summary>💡 Ver solución</summary>

```python
res = MMSimulation(AvellanedaStoikov('BTC', gamma=0.1, sigma=0.5, horizon=400), steps=400).run()
max_inv = res.max_inventory
pnl = res.final_pnl
```

</details>

### 6. El skew reduce el inventario

Con la misma semilla, simula un MarketMaker con skew (2.0) y otro sin skew (0.0), half_spread 0.3. Guarda `inv_skew` e `inv_noskew` (max inventario). El skew debe dejar menos inventario.

<sub>practicas: comparar con / sin skew</sub>

In [ ]:
from exchange.strategies import MarketMaker
from exchange.simulation import MMSimulation
inv_skew = None
inv_noskew = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert inv_skew <= inv_noskew + 1e-9, 'el skew empuja el inventario hacia 0'
print('ok  skew=%.3f noskew=%.3f' % (inv_skew, inv_noskew))

<details>
<summary>💡 Ver solución</summary>

```python
inv_skew = MMSimulation(MarketMaker('BTC', half_spread=0.3, inventory_skew=2.0), steps=400).run().max_inventory
inv_noskew = MMSimulation(MarketMaker('BTC', half_spread=0.3, inventory_skew=0.0), steps=400).run().max_inventory
```

</details>

### 7. Más gamma, más inclina el reservation price

Con inventario fijo (5), mide cuánto se aleja el reservation price del mid para gamma 0.05 y 0.8. Guarda `skew_low_gamma` y `skew_high_gamma` (= |r - mid|).

<sub>practicas: trade-off riesgo/PnL</sub>

In [ ]:
from exchange.strategies import AvellanedaStoikov
def skew_mag(g):
    s = AvellanedaStoikov('BTC', gamma=g, sigma=10, kappa=1.5, horizon=500)
    s._inventory = 5; s._t = 0
    return abs(s.reservation_price(100) - 100)
skew_low_gamma = None
skew_high_gamma = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert skew_high_gamma > skew_low_gamma, 'más gamma = más aversión = más inclinación'
print('ok  g=0.05 -> %.1f | g=0.8 -> %.1f' % (skew_low_gamma, skew_high_gamma))

<details>
<summary>💡 Ver solución</summary>

```python
skew_low_gamma = skew_mag(0.05)
skew_high_gamma = skew_mag(0.8)
```

</details>

## Cierre

El reservation price inclina según inventario y tiempo; el optimal spread cobra por el riesgo. Más gamma = más defensivo, menos inventario, menos PnL.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.